# Reproduza, teste e entregue o projeto

Uma colega precisa sair de uma pasta limpa e obter os mesmos resultados. Abra o caderno e rode desde a preparação ou extraia o kit em pasta nova. Documente a versão, a origem sintética, a ordem de execução e os nomes dos arquivos de saída. Prefira parâmetros explícitos e confirme com testes. No navegador, baixe o caderno editável e a pasta de resultados arquivo por arquivo. No terminal, reúna os arquivos em sua pasta de projeto.

Prove três propriedades. Reordenar as linhas brutas preserva o resultado. Mudar somente o km ou o alvo do teste preserva os parâmetros do treino. Trocar as datas de V06 e V07 muda a partição e os valores aprendidos. Um ID conflitante deve produzir erro com a chave. Consulte as instruções da solução em LEIA-ME-PROJETO.md.

### Rubrica (100 pontos)

Avalie cinco itens com 0, 10 ou 20 pontos. Use 0 para ausente ou contraditório, 10 para parcial e 20 para evidência completa.

1. Contagens e decisões auditáveis, com IDs e unidades.
2. Partições cronológicas e atributos disponíveis antes da partida.
3. Imputação e escala ajustadas no treino, parâmetros persistidos e saídas conferidas.
4. Referência, MAE, gráfico, tabela e limites explicados.
5. Reprodução limpa e testes dos casos de borda.

Meta para seguir a machine learning: 80 pontos, com 20 nos itens 2 e 3.

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-reproduzir-', dir=_raiz))
import numpy
import pandas
import matplotlib


## E1 · Preveja e execute

Execute o projeto do início com os dados originais e registre o resultado.

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_arquivo = _pasta / 'preparacao_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from statistics import median\n\nimport numpy as np\nimport pandas as pd\n\nATRIBUTOS = ["km_escalados", "paradas_planejadas"]\nALVO = "minutos_reais"\nFIM_TREINO = pd.Timestamp("2026-01-06")\nFIM_VALIDACAO = pd.Timestamp("2026-01-08")\n\n\ndef particionar(validas):\n    ordenadas = validas.sort_values(["data", "id"])\n    datas = ordenadas["data"]\n    partes = {\n        "treino": ordenadas.loc[datas <= FIM_TREINO],\n        "validacao": ordenadas.loc[(datas > FIM_TREINO) & (datas <= FIM_VALIDACAO)],\n        "teste": ordenadas.loc[datas > FIM_VALIDACAO],\n    }\n    if any(parte.empty for parte in partes.values()):\n        raise ValueError("treino, validação e teste devem ter pelo menos uma viagem")\n    return partes\n\n\ndef ajustar(treino):\n    km = treino["km_planejados"]\n    observados = km.dropna()\n    if observados.empty:\n        raise ValueError("treino sem distâncias observadas")\n    mediana = float(observados.median())\n    preenchidos = km.fillna(mediana)\n    return {\n        "km": {\n            "mediana": mediana,\n            "minimo": float(preenchidos.min()),\n            "maximo": float(preenchidos.max()),\n        },\n        "referencia_minutos": float(median(treino[ALVO])),\n        "atributos": ATRIBUTOS.copy(),\n        "alvo": ALVO,\n        "convencao_amplitude_zero": "km_escalados vale 0 em todas as partições",\n        "fronteiras_inclusivas": {\n            "fim_treino": FIM_TREINO.date().isoformat(),\n            "fim_validacao": FIM_VALIDACAO.date().isoformat(),\n        },\n        "instante_previsao": "antes da partida",\n        "fora_de_X": {\n            "id": "chave de rastreio",\n            "data": "define partições e fica para rastreio",\n            "estacao": "chave para auditoria",\n            "nome": "nome para auditoria",\n            "minutos_reais": "alvo conhecido depois da chegada",\n            "minutos_faturados": "calculado após a chegada",\n        },\n    }\n\n\ndef aplicar(parte, parametros):\n    escala = parametros["km"]\n    km = parte["km_planejados"].fillna(escala["mediana"])\n    amplitude = escala["maximo"] - escala["minimo"]\n    escalados = (km - escala["minimo"]) / amplitude if amplitude else pd.Series(0.0, index=parte.index)\n    return parte.assign(km_escalados=escalados)[["id", *ATRIBUTOS, ALVO]].reset_index(drop=True)\n\n\ndef extrair(tabela):\n    X = tabela[ATRIBUTOS].to_numpy(dtype=float)\n    y = tabela[ALVO].to_numpy(dtype=float)\n    ids = tabela["id"].to_numpy(dtype=str)\n    return X, y, ids\n\n\ndef exportar(partes, parametros, saida):\n    tabelas = {}\n    rastreio = []\n    for nome, parte in partes.items():\n        tabela = aplicar(parte, parametros)\n        tabelas[nome] = tabela\n        tabela.to_csv(saida / f"{nome}.csv", index=False)\n        rastreio.append(parte[["id", "data"]].assign(particao=nome))\n    pd.concat(rastreio).to_csv(saida / "rastreio.csv", index=False, date_format="%Y-%m-%d")\n    return tabelas\n\n\ndef mae(alvos, previsao):\n    if len(alvos) == 0:\n        raise ValueError("MAE requer ao menos um alvo")\n    return float(np.abs(np.asarray(alvos, dtype=float) - previsao).mean())\n', encoding='utf-8')
_arquivo = _pasta / 'relatorio_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nfrom preparacao_viagens import extrair, mae\n\n\ndef publicar_relatorio(tabelas, parametros, saida):\n    referencia = parametros["referencia_minutos"]\n    _, treino, _ = extrair(tabelas["treino"])\n    metricas = {}\n    for nome in ("validacao", "teste"):\n        _, alvos, _ = extrair(tabelas[nome])\n        metricas[nome] = mae(alvos, referencia)\n\n    fig, ax = plt.subplots(layout="constrained")\n    contagens, _, _ = ax.hist(treino, bins=4, edgecolor="black")\n    ax.set(xlabel="Duração real (min)", ylabel="Viagens de treino (contagem)",\n           title="Distribuição da duração no treino")\n    fig.savefig(saida / "duracoes_treino.png")\n    plt.close(fig)\n\n    alturas = ", ".join(str(int(contagem)) for contagem in contagens)\n    linhas = [\n        "# Duração de viagens",\n        "",\n        "Objetivo: estimar duração em minutos antes da partida usando dados sintéticos.",\n        "",\n        f"A referência prevê {referencia:.1f} min para cada viagem, mediana do alvo no treino.",\n        "A imputação e a escala também foram calculadas apenas com o treino.",\n        "A referência ficou congelada antes da avaliação de teste.",\n        "",\n        "| Partição | Viagens | MAE |",\n        "|---|---:|---:|",\n        f"| Validação | {len(tabelas[\'validacao\'])} | {metricas[\'validacao\']:.1f} min |",\n        f"| Teste | {len(tabelas[\'teste\'])} | {metricas[\'teste\']:.1f} min |",\n        "",\n        f"![Histograma de duração no treino com quatro barras de alturas {alturas}.](duracoes_treino.png)",\n        "",\n        "| Viagem | Duração real no treino (min) |",\n        "|---|---:|",\n    ]\n    for chave, duracao in tabelas["treino"][["id", "minutos_reais"]].itertuples(index=False):\n        linhas.append(f"| {chave} | {duracao:g} |")\n    linhas.extend([\n        "",\n        "O MAE é a média dos erros absolutos em minutos. Cada linha tem o mesmo peso.",\n        "Os resultados descrevem estas poucas viagens e podem variar muito com novos dias.",\n        "O gráfico foi produzido só com dados de treino, e as estações são usadas para auditoria.",\n        "",\n    ])\n    (saida / "relatorio.md").write_text("\\n".join(linhas), encoding="utf-8")\n    return metricas\n', encoding='utf-8')
_arquivo = _pasta / 'executar_projeto.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from pathlib import Path\n\nfrom auditoria_viagens import auditar, ler_estacoes, ler_viagens, salvar_json\nfrom preparacao_viagens import ajustar, exportar, extrair, particionar\nfrom relatorio_viagens import publicar_relatorio\n\n\ndef executar(viagens_csv, estacoes_csv, saida):\n    validas, auditoria = auditar(ler_viagens(viagens_csv), ler_estacoes(estacoes_csv))\n    partes = particionar(validas)\n    parametros = ajustar(partes["treino"])\n    saida.mkdir(exist_ok=True, parents=True)\n    salvar_json(saida / "auditoria.json", auditoria)\n    salvar_json(saida / "parametros.json", parametros)\n    tabelas = exportar(partes, parametros, saida)\n    metricas = publicar_relatorio(tabelas, parametros, saida)\n    return auditoria, parametros, tabelas, metricas\n\n\ndef mostrar(auditoria, parametros, tabelas, metricas):\n    print(f"Auditoria: {auditoria[\'linhas_brutas\']} brutas, {auditoria[\'linhas_validas\']} válidas.")\n    print(f"Mediana de km no treino: {parametros[\'km\'][\'mediana\']:.1f}")\n    for nome, tabela in tabelas.items():\n        X, y, ids = extrair(tabela)\n        print(nome, X.shape, y.shape, ids.tolist())\n    print(f"MAE de validação: {metricas[\'validacao\']:.1f} min")\n    print(f"MAE de teste: {metricas[\'teste\']:.1f} min")\n\n\nif __name__ == "__main__":\n    pasta = Path(__file__).resolve().parent\n    resultados = executar(pasta / "dados" / "viagens.csv", pasta / "dados" / "estacoes.csv",\n                         pasta / "saida")\n    mostrar(*resultados)\n', encoding='utf-8')
_arquivo = _pasta / 'testes_projeto.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\nfrom pathlib import Path\nimport shutil\nfrom tempfile import TemporaryDirectory\nimport unittest\n\nimport numpy as np\nimport pandas as pd\n\nfrom executar_projeto import executar\nfrom preparacao_viagens import ajustar, aplicar, extrair\n\n\nclass ProjetoTest(unittest.TestCase):\n    def setUp(self):\n        temporario = TemporaryDirectory()\n        self.addCleanup(temporario.cleanup)\n        self.base = Path(temporario.name)\n        shutil.copytree(Path(__file__).resolve().parent / "dados", self.base / "dados")\n        self.saida = self.base / "saida"\n        self.estacoes = self.base / "dados" / "estacoes.csv"\n        self.viagens = self.base / "dados" / "viagens.csv"\n\n    def executar(self, nome):\n        return executar(self.viagens, self.estacoes, self.base / nome)\n\n    def dados(self):\n        return pd.read_csv(self.viagens)\n\n    def substituir(self, tabela):\n        tabela.to_csv(self.viagens, index=False)\n\n    def test_controles_e_artefatos(self):\n        auditoria, parametros, tabelas, metricas = self.executar("saida")\n        self.assertEqual(auditoria["linhas_brutas"], 12)\n        self.assertEqual(auditoria["linhas_validas"], 10)\n        self.assertEqual(auditoria["duplicatas_excedentes"], {"V04": 1})\n        self.assertEqual(auditoria["rejeitadas"], {"V11": ["km_planejados negativo"]})\n        self.assertEqual(auditoria["km_ausentes_validas"], ["V03", "V08"])\n        self.assertEqual(auditoria["viagens_sem_cadastro"], ["V08"])\n        self.assertEqual(parametros["km"], {"mediana": 4.0, "minimo": 2.0, "maximo": 8.0})\n        self.assertEqual(parametros["referencia_minutos"], 10.0)\n        self.assertEqual(metricas, {"validacao": 3.5, "teste": 10.0})\n        for nome, ids in {"treino": range(1, 7), "validacao": range(7, 9), "teste": range(9, 11)}.items():\n            ids = [f"V{i:02d}" for i in ids]\n            tabela = pd.read_csv(self.saida / f"{nome}.csv")\n            self.assertEqual(tabela["id"].tolist(), ids)\n            X, y, chaves = extrair(tabela)\n            self.assertEqual(X.shape, (len(ids), 2))\n            self.assertEqual(y.shape, (len(ids),))\n            self.assertEqual(chaves.tolist(), ids)\n        unidas = pd.concat(tabelas.values()).set_index("id")\n        for chave, esperado in {\n            "V01": [0, 0], "V03": [1 / 3, 1], "V07": [98 / 6, 1],\n            "V08": [1 / 3, 0], "V09": [0.5, 2], "V10": [7 / 6, 3],\n        }.items():\n            np.testing.assert_allclose(unidas.loc[chave, ["km_escalados", "paradas_planejadas"]].astype(float),\n                                       esperado, atol=1e-6, rtol=0)\n        dados = json.loads((self.saida / "parametros.json").read_text(encoding="utf-8"))\n        self.assertEqual(dados, parametros)\n        rastreio = pd.read_csv(self.saida / "rastreio.csv")\n        tabela = unidas.loc[rastreio["id"]]\n        self.assertEqual(tabela.index.tolist(), rastreio["id"].tolist())\n        self.assertTrue((self.saida / "duracoes_treino.png").read_bytes().startswith(b"\\x89PNG\\r\\n\\x1a\\n"))\n        relatorio = (self.saida / "relatorio.md").read_text(encoding="utf-8")\n        self.assertIn("duracoes_treino.png", relatorio)\n        self.assertIn("3.5 min", relatorio)\n        self.assertIn("10.0 min", relatorio)\n\n    def test_reordenar_preserva_parametros_e_particoes(self):\n        _, ref, tabelas_ref, _ = self.executar("original")\n        self.substituir(self.dados().iloc[::-1])\n        _, params, tabelas, _ = self.executar("reordenado")\n        self.assertEqual(params, ref)\n        for nome in tabelas:\n            pd.testing.assert_frame_equal(tabelas[nome], tabelas_ref[nome])\n\n    def test_mudancas_so_no_teste(self):\n        _, ref, tabelas_ref, _ = self.executar("original")\n        orig = self.dados()\n        for coluna, valor in (("km_planejados", 50), ("minutos_reais", 50)):\n            with self.subTest(coluna=coluna):\n                dados = orig.copy()\n                dados.loc[dados["id"] == "V09", coluna] = valor\n                self.substituir(dados)\n                _, params, tabelas, metricas = self.executar(coluna)\n                self.assertEqual(params, ref)\n                self.assertEqual((self.base / coluna / "parametros.json").read_bytes(),\n                                 (self.base / "original" / "parametros.json").read_bytes())\n                for nome in ("treino", "validacao"):\n                    pd.testing.assert_frame_equal(tabelas[nome], tabelas_ref[nome])\n                self.assertEqual(tabelas["teste"]["id"].tolist(), ["V09", "V10"])\n                linha = tabelas["teste"].iloc[0]\n                if coluna == "km_planejados":\n                    self.assertAlmostEqual(linha["km_escalados"], 8, delta=1e-6)\n                    self.assertEqual(metricas["teste"], 10.0)\n                else:\n                    self.assertEqual(linha["minutos_reais"], 50)\n                    self.assertEqual(metricas["teste"], 26.0)\n\n    def test_troca_de_datas_muda_particoes_e_parametros(self):\n        dados = self.dados()\n        datas = dados.set_index("id")["data"]\n        dados.loc[dados["id"] == "V06", "data"] = datas["V07"]\n        dados.loc[dados["id"] == "V07", "data"] = datas["V06"]\n        self.substituir(dados)\n        _, params, tabelas, _ = self.executar("trocadas")\n        self.assertEqual(tabelas["treino"]["id"].iloc[-1], "V07")\n        self.assertEqual(tabelas["validacao"]["id"].tolist(), ["V06", "V08"])\n        self.assertEqual(params["km"]["mediana"], 4.0)\n        self.assertEqual(params["km"]["maximo"], 100.0)\n        self.assertEqual(params["referencia_minutos"], 9.0)\n\n    def test_conflito_de_id_e_erro(self):\n        dados = self.dados()\n        conflito = dados.iloc[[0]].assign(km_planejados=99)\n        self.substituir(pd.concat([dados, conflito], ignore_index=True))\n        with self.assertRaisesRegex(ValueError, "ID em conflito: V01"):\n            self.executar("conflito")\n        self.assertFalse((self.base / "conflito").exists())\n\n    def test_escala_constante_e_todos_ausentes(self):\n        dados = self.dados().iloc[:6].assign(km_planejados=4.0)\n        params = ajustar(dados)\n        self.assertTrue(aplicar(dados.assign(km_planejados=10.0), params)["km_escalados"].eq(0).all())\n        with self.assertRaisesRegex(ValueError, "treino sem distâncias observadas"):\n            ajustar(dados.assign(km_planejados=float("nan")))\n\n    def test_alvo_obrigatorio_e_nao_negativo(self):\n        dados = self.dados()\n        dados.loc[dados["id"] == "V01", "minutos_reais"] = float("nan")\n        dados.loc[dados["id"] == "V02", "minutos_reais"] = -2\n        self.substituir(dados)\n        auditoria, _, tabelas, _ = self.executar("alvos_invalidos")\n        self.assertEqual(auditoria["rejeitadas"]["V01"], ["minutos_reais ausente"])\n        self.assertEqual(auditoria["rejeitadas"]["V02"], ["minutos_reais negativo"])\n        self.assertEqual(tabelas["treino"]["id"].tolist(), ["V03", "V04", "V05", "V06"])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')
_arquivo = _pasta / 'LEIA-ME-PROJETO.md'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('# Projeto de conclusão · Duração de viagens\n\nAs viagens e estações deste projeto são sintéticas. Os originais ficam em\n`dados/viagens.csv` e `dados/estacoes.csv`. Cada viagem representa uma observação.\nQueremos estimar sua duração em minutos antes da partida.\n\n## Terminal\n\nA partir da raiz do kit, siga a instalação de CPython 3.14 e\n`requirements.txt` no README do kit. Confira o ambiente com\n`python -c "import sys, numpy, pandas, matplotlib; print(sys.version); print(numpy.__version__, pandas.__version__, matplotlib.__version__)"`.\nUse no lugar de `python` o executável da venv indicado no README, em Bash ou\nPowerShell. Os pacotes usados neste projeto são NumPy, pandas e Matplotlib e\nsuas versões de referência estão no arquivo de requisitos do kit.\n\nRode `executar_projeto.py` com esse interpretador e o caminho para o script.\nExemplo Bash a partir da raiz do kit do projeto\n\n```bash\n.venv/bin/python aulas/reproduzir/e3-resolvido/executar_projeto.py\n.venv/bin/python aulas/reproduzir/e3-resolvido/testes_projeto.py\n```\n\nEm PowerShell\n\n```powershell\n.\\.venv\\Scripts\\python.exe aulas/reproduzir/e3-resolvido/executar_projeto.py\n.\\.venv\\Scripts\\python.exe aulas/reproduzir/e3-resolvido/testes_projeto.py\n```\n\nA pasta `saida` é criada ao lado de `executar_projeto.py`. Leia\n`relatorio.md` junto ao PNG e confira os arquivos em texto. Para experimentar\noutras entradas, faça uma cópia de `dados` e chame a função `executar` com os\ncaminhos da cópia e uma nova pasta de saída. Os testes já isolam suas cópias\nem pastas temporárias.\n\n## Navegador\n\nAbra a variante resolvida do caderno de Reproduzir e execute desde a\npreparação. A primeira célula cria uma pasta de estudo independente. Para\nconferir as versões do kernel, execute a linha de Python acima depois de\nretirar `python -c` e as aspas externas. As células preparam os CSVs e módulos\nauxiliares, executam o projeto, fazem os testes e mostram o PNG. No painel de\narquivos, abra a pasta `python-reproduzir-…/e3/saida` e baixe os arquivos que\nquer guardar. Baixe também o caderno editável e guarde juntos relatório e\nfigura. Para reler um módulo que você editou, salve o arquivo e execute novamente a\npreparação dessa etapa. A preparação inicial cria outra pasta.\n\n## Arquivos produzidos\n\n- `auditoria.json` registra regras, unidades e IDs afetados.\n- `parametros.json` congela fronteiras, ordem de atributos, imputação,\n  escala, convenção de amplitude zero e referência.\n- `treino.csv`, `validacao.csv` e `teste.csv` mantêm id para alinhar atributos e alvo.\n- `rastreio.csv` preserva data e partição por ID.\n- `relatorio.md` e `duracoes_treino.png` mostram as métricas em minutos e\n  a distribuição exploratória do alvo de treino.\n\n### Rubrica (100 pontos)\n\nAvalie cinco itens com 0, 10 ou 20 pontos. Use 0 para ausente ou contraditório, 10 para parcial e 20 para evidência completa.\n\n1. Contagens e decisões auditáveis, com IDs e unidades.\n2. Partições cronológicas e atributos disponíveis antes da partida.\n3. Imputação e escala ajustadas no treino, parâmetros persistidos e saídas conferidas.\n4. Referência, MAE, gráfico, tabela e limites explicados.\n5. Reprodução limpa e testes dos casos de borda.\n\nMeta para seguir a machine learning: 80 pontos, com 20 nos itens 2 e 3.\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from executar_projeto import executar, mostrar

pasta = Path.cwd()
resultados = executar(pasta/"dados/viagens.csv", pasta/"dados/estacoes.csv", pasta/"saida")
mostrar(*resultados)


In [ ]:
from IPython.display import Image, display
display(Image(filename='saida/duracoes_treino.png'))


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
Auditoria: 12 brutas, 10 válidas.
Mediana de km no treino: 4.0
treino (6, 2) (6,) ['V01', 'V02', 'V03', 'V04', 'V05', 'V06']
validacao (2, 2) (2,) ['V07', 'V08']
teste (2, 2) (2,) ['V09', 'V10']
MAE de validação: 3.5 min
MAE de teste: 10.0 min
```

O terminal pode rodar executar_projeto.py na solução, conforme o guia do kit. O caderno reproduz as mesmas etapas e a figura. Confira os arquivos em saida.


## E2 · Altere ou depure

Adicione uma linha com V01 divergente a uma cópia da tabela e confira o diagnóstico.

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from auditoria_viagens import auditar, ler_estacoes, ler_viagens

brutas = ler_viagens(Path("dados/viagens.csv"))
estacoes = ler_estacoes(Path("dados/estacoes.csv"))
validas, auditoria = auditar(brutas, estacoes)

import pandas as pd
mudada = brutas.loc[brutas["id"].eq("V01")].assign(km_planejados=99.0)
try:
    auditar(pd.concat([brutas, mudada], ignore_index=True), estacoes)
except ValueError as erro:
    print(erro)
else:
    raise AssertionError("o conflito deveria interromper a auditoria")


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
ID em conflito: V01
```

Não tente adivinhar qual registro representa a viagem. A chamada original produz suas saídas em uma pasta independente.


## E3 · Resolva

Execute os testes, troque as datas de V06 e V07 em uma cópia e explique como X, y e os parâmetros irão chegar a um futuro modelo.

**Pista.** Rode primeiro sem alterações. Use uma cópia para as variações e guarde os JSONs de parâmetros para comparar.

Saída esperada

```text
7 testes do projeto passaram
Auditoria: 12 brutas, 10 válidas.
Mediana de km no treino: 4.0
treino (6, 2) (6,) ['V01', 'V02', 'V03', 'V04', 'V05', 'V06']
validacao (2, 2) (2,) ['V07', 'V08']
teste (2, 2) (2,) ['V09', 'V10']
MAE de validação: 3.5 min
MAE de teste: 10.0 min
```

Se editar um auxiliar no painel de arquivos, salve a edição. Execute novamente a preparação desta etapa e a célula do exercício. A preparação inicial do caderno cria outra pasta.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'dados/estacoes.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('estacao,nome\nA,Centro\nB,Norte\n', encoding='utf-8')
_arquivo = _pasta / 'dados/viagens.csv'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('id,data,estacao,km_planejados,paradas_planejadas,minutos_reais,minutos_faturados\nV01,2026-01-01,A,2,0,6,6\nV02,2026-01-02,B,4,1,8,8\nV03,2026-01-03,A,,1,10,10\nV04,2026-01-04,B,6,2,10,10\nV05,2026-01-05,A,4,2,12,12\nV06,2026-01-06,B,8,3,14,14\nV07,2026-01-07,A,100,1,8,8\nV08,2026-01-08,C,,0,15,15\nV09,2026-01-09,B,5,2,18,18\nV10,2026-01-10,A,9,3,22,22\nV11,2026-01-11,A,-1,1,7,7\nV04,2026-01-04,B,6,2,10,10\n', encoding='utf-8')
_arquivo = _pasta / 'auditoria_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\n\nimport numpy as np\nimport pandas as pd\n\nTIPOS = {\n    "id": "string",\n    "data": "string",\n    "estacao": "string",\n    "km_planejados": "float64",\n    "paradas_planejadas": "Int64",\n    "minutos_reais": "float64",\n    "minutos_faturados": "float64",\n}\n\nUNIDADES = {\n    "id": "chave de viagem",\n    "data": "dia (AAAA-MM-DD)",\n    "estacao": "chave de estação",\n    "km_planejados": "km",\n    "paradas_planejadas": "paradas (contagem)",\n    "minutos_reais": "min",\n    "minutos_faturados": "min",\n}\n\n\ndef ler_viagens(caminho):\n    viagens = pd.read_csv(caminho, dtype=TIPOS, keep_default_na=False, na_values=[""])\n    if set(viagens.columns) != set(TIPOS):\n        raise ValueError("colunas de viagens diferentes do esquema")\n    if viagens[["id", "data", "estacao"]].isna().any().any():\n        raise ValueError("id, data e estacao devem ser preenchidos")\n    if not viagens["data"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}").all():\n        raise ValueError("data deve estar no formato AAAA-MM-DD")\n    viagens["data"] = pd.to_datetime(viagens["data"], format="%Y-%m-%d")\n    return viagens\n\n\ndef ler_estacoes(caminho):\n    estacoes = pd.read_csv(caminho, dtype="string", keep_default_na=False, na_values=[""])\n    if set(estacoes.columns) != {"estacao", "nome"} or estacoes.isna().any().any():\n        raise ValueError("estacoes.csv deve ter chave e nome preenchidos")\n    return estacoes\n\n\ndef auditar(brutas, estacoes):\n    duplicatas = brutas.loc[brutas.duplicated(), "id"].value_counts().sort_index()\n    unicas = brutas.drop_duplicates()\n    conflitos = sorted(unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique())\n    if conflitos:\n        prefixo = "ID em conflito: " if len(conflitos) == 1 else "IDs em conflito: "\n        raise ValueError(prefixo + ", ".join(conflitos))\n\n    regras = {\n        "km_planejados negativo": unicas["km_planejados"].lt(0),\n        "km_planejados infinito": np.isinf(unicas["km_planejados"]),\n        "paradas_planejadas ausente": unicas["paradas_planejadas"].isna(),\n        "paradas_planejadas negativo": unicas["paradas_planejadas"].lt(0).fillna(False),\n        "minutos_reais ausente": unicas["minutos_reais"].isna(),\n        "minutos_reais negativo": unicas["minutos_reais"].lt(0),\n        "minutos_reais infinito": np.isinf(unicas["minutos_reais"]),\n    }\n    rejeitadas = {}\n    for regra, mascara in regras.items():\n        for chave in sorted(unicas.loc[mascara, "id"]):\n            rejeitadas.setdefault(chave, []).append(regra)\n    validas = unicas.loc[~unicas["id"].isin(rejeitadas)].copy()\n    validas["paradas_planejadas"] = validas["paradas_planejadas"].astype("int64")\n    uniao = validas.merge(estacoes, on="estacao", how="left", validate="many_to_one", indicator=True)\n    auditoria = {\n        "linhas_brutas": len(brutas),\n        "linhas_validas": len(uniao),\n        "duplicatas_excedentes": {chave: int(n) for chave, n in duplicatas.items()},\n        "total_duplicatas_excedentes": int(duplicatas.sum()),\n        "rejeitadas": rejeitadas,\n        "total_rejeitadas": len(rejeitadas),\n        "km_ausentes_validas": sorted(validas.loc[validas["km_planejados"].isna(), "id"]),\n        "viagens_sem_cadastro": sorted(uniao.loc[uniao["_merge"] == "left_only", "id"]),\n        "decisoes": [\n            "Conservar o original e remover só cópias de linhas idênticas.",\n            "Interromper quando registros diferentes usam o mesmo id.",\n            "Conservar km ausente para imputação no preparo.",\n            "Conservar zero paradas e distâncias altas dentro do domínio.",\n            "Manter em auditoria as viagens sem cadastro de estação.",\n        ],\n        "unidades": UNIDADES,\n    }\n    return uniao.drop(columns="_merge"), auditoria\n\n\ndef salvar_json(caminho, dados):\n    caminho.write_text(\n        json.dumps(dados, ensure_ascii=False, indent=2, sort_keys=True) + "\\n",\n        encoding="utf-8",\n    )\n', encoding='utf-8')
_arquivo = _pasta / 'preparacao_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from statistics import median\n\nimport numpy as np\nimport pandas as pd\n\nATRIBUTOS = ["km_escalados", "paradas_planejadas"]\nALVO = "minutos_reais"\nFIM_TREINO = pd.Timestamp("2026-01-06")\nFIM_VALIDACAO = pd.Timestamp("2026-01-08")\n\n\ndef particionar(validas):\n    ordenadas = validas.sort_values(["data", "id"])\n    datas = ordenadas["data"]\n    partes = {\n        "treino": ordenadas.loc[datas <= FIM_TREINO],\n        "validacao": ordenadas.loc[(datas > FIM_TREINO) & (datas <= FIM_VALIDACAO)],\n        "teste": ordenadas.loc[datas > FIM_VALIDACAO],\n    }\n    if any(parte.empty for parte in partes.values()):\n        raise ValueError("treino, validação e teste devem ter pelo menos uma viagem")\n    return partes\n\n\ndef ajustar(treino):\n    km = treino["km_planejados"]\n    observados = km.dropna()\n    if observados.empty:\n        raise ValueError("treino sem distâncias observadas")\n    mediana = float(observados.median())\n    preenchidos = km.fillna(mediana)\n    return {\n        "km": {\n            "mediana": mediana,\n            "minimo": float(preenchidos.min()),\n            "maximo": float(preenchidos.max()),\n        },\n        "referencia_minutos": float(median(treino[ALVO])),\n        "atributos": ATRIBUTOS.copy(),\n        "alvo": ALVO,\n        "convencao_amplitude_zero": "km_escalados vale 0 em todas as partições",\n        "fronteiras_inclusivas": {\n            "fim_treino": FIM_TREINO.date().isoformat(),\n            "fim_validacao": FIM_VALIDACAO.date().isoformat(),\n        },\n        "instante_previsao": "antes da partida",\n        "fora_de_X": {\n            "id": "chave de rastreio",\n            "data": "define partições e fica para rastreio",\n            "estacao": "chave para auditoria",\n            "nome": "nome para auditoria",\n            "minutos_reais": "alvo conhecido depois da chegada",\n            "minutos_faturados": "calculado após a chegada",\n        },\n    }\n\n\ndef aplicar(parte, parametros):\n    escala = parametros["km"]\n    km = parte["km_planejados"].fillna(escala["mediana"])\n    amplitude = escala["maximo"] - escala["minimo"]\n    escalados = (km - escala["minimo"]) / amplitude if amplitude else pd.Series(0.0, index=parte.index)\n    return parte.assign(km_escalados=escalados)[["id", *ATRIBUTOS, ALVO]].reset_index(drop=True)\n\n\ndef extrair(tabela):\n    X = tabela[ATRIBUTOS].to_numpy(dtype=float)\n    y = tabela[ALVO].to_numpy(dtype=float)\n    ids = tabela["id"].to_numpy(dtype=str)\n    return X, y, ids\n\n\ndef exportar(partes, parametros, saida):\n    tabelas = {}\n    rastreio = []\n    for nome, parte in partes.items():\n        tabela = aplicar(parte, parametros)\n        tabelas[nome] = tabela\n        tabela.to_csv(saida / f"{nome}.csv", index=False)\n        rastreio.append(parte[["id", "data"]].assign(particao=nome))\n    pd.concat(rastreio).to_csv(saida / "rastreio.csv", index=False, date_format="%Y-%m-%d")\n    return tabelas\n\n\ndef mae(alvos, previsao):\n    if len(alvos) == 0:\n        raise ValueError("MAE requer ao menos um alvo")\n    return float(np.abs(np.asarray(alvos, dtype=float) - previsao).mean())\n', encoding='utf-8')
_arquivo = _pasta / 'relatorio_viagens.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\nfrom preparacao_viagens import extrair, mae\n\n\ndef publicar_relatorio(tabelas, parametros, saida):\n    referencia = parametros["referencia_minutos"]\n    _, treino, _ = extrair(tabelas["treino"])\n    metricas = {}\n    for nome in ("validacao", "teste"):\n        _, alvos, _ = extrair(tabelas[nome])\n        metricas[nome] = mae(alvos, referencia)\n\n    fig, ax = plt.subplots(layout="constrained")\n    contagens, _, _ = ax.hist(treino, bins=4, edgecolor="black")\n    ax.set(xlabel="Duração real (min)", ylabel="Viagens de treino (contagem)",\n           title="Distribuição da duração no treino")\n    fig.savefig(saida / "duracoes_treino.png")\n    plt.close(fig)\n\n    alturas = ", ".join(str(int(contagem)) for contagem in contagens)\n    linhas = [\n        "# Duração de viagens",\n        "",\n        "Objetivo: estimar duração em minutos antes da partida usando dados sintéticos.",\n        "",\n        f"A referência prevê {referencia:.1f} min para cada viagem, mediana do alvo no treino.",\n        "A imputação e a escala também foram calculadas apenas com o treino.",\n        "A referência ficou congelada antes da avaliação de teste.",\n        "",\n        "| Partição | Viagens | MAE |",\n        "|---|---:|---:|",\n        f"| Validação | {len(tabelas[\'validacao\'])} | {metricas[\'validacao\']:.1f} min |",\n        f"| Teste | {len(tabelas[\'teste\'])} | {metricas[\'teste\']:.1f} min |",\n        "",\n        f"![Histograma de duração no treino com quatro barras de alturas {alturas}.](duracoes_treino.png)",\n        "",\n        "| Viagem | Duração real no treino (min) |",\n        "|---|---:|",\n    ]\n    for chave, duracao in tabelas["treino"][["id", "minutos_reais"]].itertuples(index=False):\n        linhas.append(f"| {chave} | {duracao:g} |")\n    linhas.extend([\n        "",\n        "O MAE é a média dos erros absolutos em minutos. Cada linha tem o mesmo peso.",\n        "Os resultados descrevem estas poucas viagens e podem variar muito com novos dias.",\n        "O gráfico foi produzido só com dados de treino, e as estações são usadas para auditoria.",\n        "",\n    ])\n    (saida / "relatorio.md").write_text("\\n".join(linhas), encoding="utf-8")\n    return metricas\n', encoding='utf-8')
_arquivo = _pasta / 'executar_projeto.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('from pathlib import Path\n\nfrom auditoria_viagens import auditar, ler_estacoes, ler_viagens, salvar_json\nfrom preparacao_viagens import ajustar, exportar, extrair, particionar\nfrom relatorio_viagens import publicar_relatorio\n\n\ndef executar(viagens_csv, estacoes_csv, saida):\n    validas, auditoria = auditar(ler_viagens(viagens_csv), ler_estacoes(estacoes_csv))\n    partes = particionar(validas)\n    parametros = ajustar(partes["treino"])\n    saida.mkdir(exist_ok=True, parents=True)\n    salvar_json(saida / "auditoria.json", auditoria)\n    salvar_json(saida / "parametros.json", parametros)\n    tabelas = exportar(partes, parametros, saida)\n    metricas = publicar_relatorio(tabelas, parametros, saida)\n    return auditoria, parametros, tabelas, metricas\n\n\ndef mostrar(auditoria, parametros, tabelas, metricas):\n    print(f"Auditoria: {auditoria[\'linhas_brutas\']} brutas, {auditoria[\'linhas_validas\']} válidas.")\n    print(f"Mediana de km no treino: {parametros[\'km\'][\'mediana\']:.1f}")\n    for nome, tabela in tabelas.items():\n        X, y, ids = extrair(tabela)\n        print(nome, X.shape, y.shape, ids.tolist())\n    print(f"MAE de validação: {metricas[\'validacao\']:.1f} min")\n    print(f"MAE de teste: {metricas[\'teste\']:.1f} min")\n\n\nif __name__ == "__main__":\n    pasta = Path(__file__).resolve().parent\n    resultados = executar(pasta / "dados" / "viagens.csv", pasta / "dados" / "estacoes.csv",\n                         pasta / "saida")\n    mostrar(*resultados)\n', encoding='utf-8')
_arquivo = _pasta / 'testes_projeto.py'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('import json\nfrom pathlib import Path\nimport shutil\nfrom tempfile import TemporaryDirectory\nimport unittest\n\nimport numpy as np\nimport pandas as pd\n\nfrom executar_projeto import executar\nfrom preparacao_viagens import ajustar, aplicar, extrair\n\n\nclass ProjetoTest(unittest.TestCase):\n    def setUp(self):\n        temporario = TemporaryDirectory()\n        self.addCleanup(temporario.cleanup)\n        self.base = Path(temporario.name)\n        shutil.copytree(Path(__file__).resolve().parent / "dados", self.base / "dados")\n        self.saida = self.base / "saida"\n        self.estacoes = self.base / "dados" / "estacoes.csv"\n        self.viagens = self.base / "dados" / "viagens.csv"\n\n    def executar(self, nome):\n        return executar(self.viagens, self.estacoes, self.base / nome)\n\n    def dados(self):\n        return pd.read_csv(self.viagens)\n\n    def substituir(self, tabela):\n        tabela.to_csv(self.viagens, index=False)\n\n    def test_controles_e_artefatos(self):\n        auditoria, parametros, tabelas, metricas = self.executar("saida")\n        self.assertEqual(auditoria["linhas_brutas"], 12)\n        self.assertEqual(auditoria["linhas_validas"], 10)\n        self.assertEqual(auditoria["duplicatas_excedentes"], {"V04": 1})\n        self.assertEqual(auditoria["rejeitadas"], {"V11": ["km_planejados negativo"]})\n        self.assertEqual(auditoria["km_ausentes_validas"], ["V03", "V08"])\n        self.assertEqual(auditoria["viagens_sem_cadastro"], ["V08"])\n        self.assertEqual(parametros["km"], {"mediana": 4.0, "minimo": 2.0, "maximo": 8.0})\n        self.assertEqual(parametros["referencia_minutos"], 10.0)\n        self.assertEqual(metricas, {"validacao": 3.5, "teste": 10.0})\n        for nome, ids in {"treino": range(1, 7), "validacao": range(7, 9), "teste": range(9, 11)}.items():\n            ids = [f"V{i:02d}" for i in ids]\n            tabela = pd.read_csv(self.saida / f"{nome}.csv")\n            self.assertEqual(tabela["id"].tolist(), ids)\n            X, y, chaves = extrair(tabela)\n            self.assertEqual(X.shape, (len(ids), 2))\n            self.assertEqual(y.shape, (len(ids),))\n            self.assertEqual(chaves.tolist(), ids)\n        unidas = pd.concat(tabelas.values()).set_index("id")\n        for chave, esperado in {\n            "V01": [0, 0], "V03": [1 / 3, 1], "V07": [98 / 6, 1],\n            "V08": [1 / 3, 0], "V09": [0.5, 2], "V10": [7 / 6, 3],\n        }.items():\n            np.testing.assert_allclose(unidas.loc[chave, ["km_escalados", "paradas_planejadas"]].astype(float),\n                                       esperado, atol=1e-6, rtol=0)\n        dados = json.loads((self.saida / "parametros.json").read_text(encoding="utf-8"))\n        self.assertEqual(dados, parametros)\n        rastreio = pd.read_csv(self.saida / "rastreio.csv")\n        tabela = unidas.loc[rastreio["id"]]\n        self.assertEqual(tabela.index.tolist(), rastreio["id"].tolist())\n        self.assertTrue((self.saida / "duracoes_treino.png").read_bytes().startswith(b"\\x89PNG\\r\\n\\x1a\\n"))\n        relatorio = (self.saida / "relatorio.md").read_text(encoding="utf-8")\n        self.assertIn("duracoes_treino.png", relatorio)\n        self.assertIn("3.5 min", relatorio)\n        self.assertIn("10.0 min", relatorio)\n\n    def test_reordenar_preserva_parametros_e_particoes(self):\n        _, ref, tabelas_ref, _ = self.executar("original")\n        self.substituir(self.dados().iloc[::-1])\n        _, params, tabelas, _ = self.executar("reordenado")\n        self.assertEqual(params, ref)\n        for nome in tabelas:\n            pd.testing.assert_frame_equal(tabelas[nome], tabelas_ref[nome])\n\n    def test_mudancas_so_no_teste(self):\n        _, ref, tabelas_ref, _ = self.executar("original")\n        orig = self.dados()\n        for coluna, valor in (("km_planejados", 50), ("minutos_reais", 50)):\n            with self.subTest(coluna=coluna):\n                dados = orig.copy()\n                dados.loc[dados["id"] == "V09", coluna] = valor\n                self.substituir(dados)\n                _, params, tabelas, metricas = self.executar(coluna)\n                self.assertEqual(params, ref)\n                self.assertEqual((self.base / coluna / "parametros.json").read_bytes(),\n                                 (self.base / "original" / "parametros.json").read_bytes())\n                for nome in ("treino", "validacao"):\n                    pd.testing.assert_frame_equal(tabelas[nome], tabelas_ref[nome])\n                self.assertEqual(tabelas["teste"]["id"].tolist(), ["V09", "V10"])\n                linha = tabelas["teste"].iloc[0]\n                if coluna == "km_planejados":\n                    self.assertAlmostEqual(linha["km_escalados"], 8, delta=1e-6)\n                    self.assertEqual(metricas["teste"], 10.0)\n                else:\n                    self.assertEqual(linha["minutos_reais"], 50)\n                    self.assertEqual(metricas["teste"], 26.0)\n\n    def test_troca_de_datas_muda_particoes_e_parametros(self):\n        dados = self.dados()\n        datas = dados.set_index("id")["data"]\n        dados.loc[dados["id"] == "V06", "data"] = datas["V07"]\n        dados.loc[dados["id"] == "V07", "data"] = datas["V06"]\n        self.substituir(dados)\n        _, params, tabelas, _ = self.executar("trocadas")\n        self.assertEqual(tabelas["treino"]["id"].iloc[-1], "V07")\n        self.assertEqual(tabelas["validacao"]["id"].tolist(), ["V06", "V08"])\n        self.assertEqual(params["km"]["mediana"], 4.0)\n        self.assertEqual(params["km"]["maximo"], 100.0)\n        self.assertEqual(params["referencia_minutos"], 9.0)\n\n    def test_conflito_de_id_e_erro(self):\n        dados = self.dados()\n        conflito = dados.iloc[[0]].assign(km_planejados=99)\n        self.substituir(pd.concat([dados, conflito], ignore_index=True))\n        with self.assertRaisesRegex(ValueError, "ID em conflito: V01"):\n            self.executar("conflito")\n        self.assertFalse((self.base / "conflito").exists())\n\n    def test_escala_constante_e_todos_ausentes(self):\n        dados = self.dados().iloc[:6].assign(km_planejados=4.0)\n        params = ajustar(dados)\n        self.assertTrue(aplicar(dados.assign(km_planejados=10.0), params)["km_escalados"].eq(0).all())\n        with self.assertRaisesRegex(ValueError, "treino sem distâncias observadas"):\n            ajustar(dados.assign(km_planejados=float("nan")))\n\n    def test_alvo_obrigatorio_e_nao_negativo(self):\n        dados = self.dados()\n        dados.loc[dados["id"] == "V01", "minutos_reais"] = float("nan")\n        dados.loc[dados["id"] == "V02", "minutos_reais"] = -2\n        self.substituir(dados)\n        auditoria, _, tabelas, _ = self.executar("alvos_invalidos")\n        self.assertEqual(auditoria["rejeitadas"]["V01"], ["minutos_reais ausente"])\n        self.assertEqual(auditoria["rejeitadas"]["V02"], ["minutos_reais negativo"])\n        self.assertEqual(tabelas["treino"]["id"].tolist(), ["V03", "V04", "V05", "V06"])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')
_arquivo = _pasta / 'LEIA-ME-PROJETO.md'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('# Projeto de conclusão · Duração de viagens\n\nAs viagens e estações deste projeto são sintéticas. Os originais ficam em\n`dados/viagens.csv` e `dados/estacoes.csv`. Cada viagem representa uma observação.\nQueremos estimar sua duração em minutos antes da partida.\n\n## Terminal\n\nA partir da raiz do kit, siga a instalação de CPython 3.14 e\n`requirements.txt` no README do kit. Confira o ambiente com\n`python -c "import sys, numpy, pandas, matplotlib; print(sys.version); print(numpy.__version__, pandas.__version__, matplotlib.__version__)"`.\nUse no lugar de `python` o executável da venv indicado no README, em Bash ou\nPowerShell. Os pacotes usados neste projeto são NumPy, pandas e Matplotlib e\nsuas versões de referência estão no arquivo de requisitos do kit.\n\nRode `executar_projeto.py` com esse interpretador e o caminho para o script.\nExemplo Bash a partir da raiz do kit do projeto\n\n```bash\n.venv/bin/python aulas/reproduzir/e3-resolvido/executar_projeto.py\n.venv/bin/python aulas/reproduzir/e3-resolvido/testes_projeto.py\n```\n\nEm PowerShell\n\n```powershell\n.\\.venv\\Scripts\\python.exe aulas/reproduzir/e3-resolvido/executar_projeto.py\n.\\.venv\\Scripts\\python.exe aulas/reproduzir/e3-resolvido/testes_projeto.py\n```\n\nA pasta `saida` é criada ao lado de `executar_projeto.py`. Leia\n`relatorio.md` junto ao PNG e confira os arquivos em texto. Para experimentar\noutras entradas, faça uma cópia de `dados` e chame a função `executar` com os\ncaminhos da cópia e uma nova pasta de saída. Os testes já isolam suas cópias\nem pastas temporárias.\n\n## Navegador\n\nAbra a variante resolvida do caderno de Reproduzir e execute desde a\npreparação. A primeira célula cria uma pasta de estudo independente. Para\nconferir as versões do kernel, execute a linha de Python acima depois de\nretirar `python -c` e as aspas externas. As células preparam os CSVs e módulos\nauxiliares, executam o projeto, fazem os testes e mostram o PNG. No painel de\narquivos, abra a pasta `python-reproduzir-…/e3/saida` e baixe os arquivos que\nquer guardar. Baixe também o caderno editável e guarde juntos relatório e\nfigura. Para reler um módulo que você editou, salve o arquivo e execute novamente a\npreparação dessa etapa. A preparação inicial cria outra pasta.\n\n## Arquivos produzidos\n\n- `auditoria.json` registra regras, unidades e IDs afetados.\n- `parametros.json` congela fronteiras, ordem de atributos, imputação,\n  escala, convenção de amplitude zero e referência.\n- `treino.csv`, `validacao.csv` e `teste.csv` mantêm id para alinhar atributos e alvo.\n- `rastreio.csv` preserva data e partição por ID.\n- `relatorio.md` e `duracoes_treino.png` mostram as métricas em minutos e\n  a distribuição exploratória do alvo de treino.\n\n### Rubrica (100 pontos)\n\nAvalie cinco itens com 0, 10 ou 20 pontos. Use 0 para ausente ou contraditório, 10 para parcial e 20 para evidência completa.\n\n1. Contagens e decisões auditáveis, com IDs e unidades.\n2. Partições cronológicas e atributos disponíveis antes da partida.\n3. Imputação e escala ajustadas no treino, parâmetros persistidos e saídas conferidas.\n4. Referência, MAE, gráfico, tabela e limites explicados.\n5. Reprodução limpa e testes dos casos de borda.\n\nMeta para seguir a machine learning: 80 pontos, com 20 nos itens 2 e 3.\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path
from executar_projeto import executar, mostrar

pasta = Path.cwd()
resultados = executar(pasta/"dados/viagens.csv", pasta/"dados/estacoes.csv", pasta/"saida")
# Use os testes do projeto e registre as variações com uma cópia dos dados.


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Compare com a solução comentada na aula e faça o caso de borda antes de concluir.
